In [0]:
import re
import json
import unicodedata
import time
import random
from datetime import datetime, timedelta
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Referer": "https://████████████/"
}

def slugify(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-zA-Z0-9]+", "-", s.lower()).strip("-")

def gerar_rotas(cidades):
    rotas = []
    for i in range(len(cidades)):
        for j in range(len(cidades)):
            if i != j:
                o, ouf = cidades[i]
                d, duf = cidades[j]
                rotas.append({
                    "origem_url": f"{slugify(o)}-{ouf.lower()}-todos",
                    "destino_url": f"{slugify(d)}-{duf.lower()}-todos"
                })
    return rotas

def fetch_state(url):
    try:
        time.sleep(random.uniform(3.0, 5.0))
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        m = re.search(r'window\.__PRELOADED_STATE__\s*=\s*(\{.*?\})\s*;\s*', r.text, flags=re.DOTALL)
        if m:
            return json.loads(m.group(1))
        return None
    except:
        return None

def extract(state, origem, destino, dt_execucao):
    rows = []
    if not state:
        return rows
    try:
        search_data = state.get("aditionalValueFromBackend", {}).get("search", {}).get("result", {})
        routes = search_data.get("routes", {}).get("departureRoutes") or []
        ts = datetime.now().isoformat()

        for r in routes:
            trips = r.get("tripList", []) or []
            preco = r.get("totalPrice") or r.get("price")
            empresas = list(dict.fromkeys([t.get("company", {}).get("friendlyName") or t.get("companyName") or "N/A" for t in trips]))

            rows.append({
                "origem_url": origem,
                "destino_url": destino,
                "data_consulta": dt_execucao,
                "data_extracao": ts,
                "data_viagem": r.get("initialDepartureDate") or r.get("departureDate"),
                "origem": r.get("initialOriginCity") or r.get("originCity"),
                "destino": r.get("finalDestinationCity") or r.get("destinationCity"),
                "preco": float(preco) if preco else None,
                "empresas_raw": json.dumps(empresas, ensure_ascii=False),
                "assentos_disponiveis_raw": json.dumps([t.get("availableSeats", 0) for t in trips]),
                "raw_data": json.dumps(r, ensure_ascii=False)
            })
    except:
        pass
    return rows

def executar_bronze(n_dias=7):
    cidades = [
        ("Vitoria", "ES"), 
        ("Rio de Janeiro", "RJ"), 
        ("Sao Paulo", "SP"), 
        ("Belo Horizonte", "MG")
    ]
    rotas = gerar_rotas(cidades)
    dados_totais = []
    base = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)

    print(f"Iniciando extração de {len(rotas)} rotas para os próximos {n_dias} dias...")

    for rota in rotas:
        for i in range(n_dias + 1):
            dt_url = (base + timedelta(days=i)).strftime("%d/%m/%Y")
            dt_log = (base + timedelta(days=i)).strftime("%Y-%m-%d")
            url = f"https://████████████████████████/{rota['origem_url']}-para-{rota['destino_url']}?departureDate={dt_url}"
            
            state = fetch_state(url)
            if state:
                viagens = extract(state, rota["origem_url"], rota["destino_url"], dt_log)
                dados_totais.extend(viagens)

    if dados_totais:
        df = spark.createDataFrame(dados_totais)
        window = Window.partitionBy("origem_url", "destino_url", "data_viagem").orderBy(F.col("data_extracao").desc())
        df_final = df.withColumn("rn", F.row_number().over(window)).filter("rn = 1").drop("rn")
        df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_passagens")
        print(f"Extração concluída com sucesso. Total de registros: {len(dados_totais)}")
    else:
        print("Nenhum dado coletado.")

executar_bronze(7)

Iniciando extração de 12 rotas para os próximos 7 dias...
Extração concluída com sucesso. Total de registros: 6211
